In [10]:
import time
import requests
import pandas as pd
import csv
from datetime import datetime

# ========== 第一部分：获取比赛排名 ==========
API_RANK = "https://ac.nowcoder.com/acm-heavy/acm/contest/real-time-rank-data"
API_STATUS = "https://ac.nowcoder.com/acm-heavy/acm/contest/status-list"

# 牛客名 -> 真实姓名
name_map = {
    "yubeibei": "徐佳凝",
    "alayaf": "刘云芳",
    "Patience019": "李新栋",
    "lihanyu": "李晗雨",
    "HHHzzzyyy": "韩郑洋",
    "轩屁屁的核桃": "张雨轩",
    "坠姆哈尼": "张晟辉",
    "大鱼儿半生不熟": "余俊颉",
    "南与江": "姚依涛",
    "枣虾鱼": "赵雨欣",
    "Makima667": "杨潇涵",
    "柯大人": "赵柯军",
    "水悲": "钱烁",
    "caimingzheng": "蔡明正",
    "牛客891390731号": "丁怡平",
    "kyhnb66": "库宇航",
    "万月838326": "胡钧轶",
    "牛客390356941号": "张印",
    "芭六": "圣永贤",
    "ekko___": "赵英浩",
    "2421211576": "王泽",
    "真想一下子睡过去": "石锦文",
    "谬儒": "穆栋威",
    "keshao": "倪典",
    "GoldenRipple": "赵文杰",
    "幼儿园校霸的校霸": "杨俊毅",
    "ccs821631": "陈长尚",
}

def fetch_rank(contest_id, limit=100000, page_size=10000):
    """获取比赛排名数据"""
    params = {
        "token": "",
        "id": contest_id,
        "page": 1,
        "pageSize": page_size,
        "limit": limit,
        "_": int(time.time() * 1000)
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": f"https://ac.nowcoder.com/acm/contest/{contest_id}",
        "X-Requested-With": "XMLHttpRequest",
    }

    r = requests.get(API_RANK, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    js = r.json()
    return js["data"]["rankData"], js["data"]["basicInfo"]


# ========== 第二部分：获取用户提交状态 ==========
def fetch_user_submissions(contest_id, username):
    """
    从牛客网API获取用户在指定比赛中的提交记录
    """
    params = {
        "token": "",
        "id": contest_id,
        "pageSize": 50,
        "searchUserName": username,
        "_": int(time.time() * 1000)
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Referer": f"https://ac.nowcoder.com/acm/contest/{contest_id}",
        "Accept": "text/plain, */*; q=0.01"
    }

    try:
        response = requests.get(API_STATUS, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"  获取用户 {username} 提交记录失败: {e}")
        return None


def get_problem_status(api_data, problem_count=13):
    """
    获取每道题的最终状态
    返回一个字典 {题目: 状态} 和 每道题的状态列表
    状态: 'AC', 'WA', ''（空字符串表示未提交）
    """
    if not api_data or api_data.get('code') != 0:
        return None, [""] * problem_count

    submissions = api_data['data']['data']

    # 初始化题目状态
    all_problems = [chr(ord('A') + i) for i in range(problem_count)]
    problem_final_status = {problem: "" for problem in all_problems}

    # 记录每道题是否AC
    ac_problems = set()

    # 先找出所有AC的题目
    for sub in submissions:
        problem_index = sub.get('index')
        status = sub.get('statusMessage', '')

        if not problem_index:
            continue

        if status == '答案正确':
            ac_problems.add(problem_index)

    # 对于非AC的题目，检查是否有WA提交
    for sub in submissions:
        problem_index = sub.get('index')
        status = sub.get('statusMessage', '')

        if not problem_index:
            continue

        if problem_index not in ac_problems:
            if status in ['答案错误', '运行错误', '编译错误', '超时', '内存超限']:
                if problem_final_status[problem_index] == "":
                    problem_final_status[problem_index] = "WA"

    # 将AC的题目标记为AC
    for problem in ac_problems:
        problem_final_status[problem] = "AC"

    # 生成状态列表（按A-M顺序）
    status_list = [problem_final_status[p] for p in all_problems]

    return problem_final_status, status_list


# ========== 第三部分：构建数据框 ==========
def build_school_df(rank_data, contest_id):
    """构建包含参赛信息和题目状态的DataFrame"""
    participants = {}

    # 题目列表
    all_problems = [chr(ord('A') + i) for i in range(13)]

    print(f"\n📋 本题比赛题目范围: {' '.join(all_problems)}")

    # 首先，记录所有在排名中的选手
    for x in rank_data:
        user = x.get("userName", "")
        if user in name_map:
            # 获取该用户的题目提交状态
            print(f"  正在获取 {user} 的提交状态...")
            api_data = fetch_user_submissions(contest_id, user)
            result = get_problem_status(api_data)

            # 基础信息
            participant_data = {
                "排名": x.get("ranking", -1),
                "真实姓名": name_map[user],
                "牛客名": user,
                "AC数(总)": x.get("acceptedCount", 0),
                "罚时(ms)": x.get("penaltyTime", 0),
                "学校": x.get("school", ""),
                "参赛状态": "已参赛",
            }

            if result and result[0]:
                status_dict, status_list = result

                # 添加每道题的状态作为单独的列（未提交显示空字符串）
                for i, problem in enumerate(all_problems):
                    participant_data[f"题{problem}"] = status_list[i]

                # 统计各状态数量
                ac_count_detail = status_list.count('AC')
                wa_count_detail = status_list.count('WA')
                unsubmitted_detail = status_list.count('')

                participant_data["AC数(详)"] = ac_count_detail
                participant_data["WA数(详)"] = wa_count_detail
                participant_data["未提交数"] = unsubmitted_detail
            else:
                # 获取失败的情况
                for problem in all_problems:
                    participant_data[f"题{problem}"] = ""
                participant_data["AC数(详)"] = 0
                participant_data["WA数(详)"] = 0
                participant_data["未提交数"] = 13

            participants[user] = participant_data

    # 然后，检查所有在name_map中但没参赛的人
    for user, real_name in name_map.items():
        if user not in participants:
            participant_data = {
                "排名": "未参赛",
                "真实姓名": real_name,
                "牛客名": user,
                "AC数(总)": 0,
                "罚时(ms)": 0,
                "学校": "",
                "参赛状态": "未参赛",
            }

            participants[user] = participant_data

    # 转换为DataFrame
    rows = list(participants.values())
    return pd.DataFrame(rows)


def save_user_status_to_csv(status_dict, username, contest_id):
    """
    保存单个用户的题目状态为CSV文件（展开为13列）
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"牛客_{username}_比赛{contest_id}_详细状态_{timestamp}.csv"

    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)

        problems = list(status_dict.keys())
        statuses = [status_dict[p] for p in problems]

        # 第一行：题号
        writer.writerow(['项目'] + problems)

        # 第二行：状态（将空字符串替换为空白）
        display_statuses = ['' if s == '' else s for s in statuses]
        writer.writerow(['状态'] + display_statuses)

        # 第三行：统计
        ac_count = statuses.count('AC')
        wa_count = statuses.count('WA')
        unsubmitted_count = statuses.count('')

        writer.writerow(['统计', f'AC:{ac_count}', f'WA:{wa_count}', f'未提交:{unsubmitted_count}'] + [''] * 10)

    return filename


# ========== 第四部分：主函数 ==========
def solve():
    # 要爬取的多个比赛ID
    contest_ids = [120561, 120562, 120563, 120564, 120565, 120566]
    # 题目列表
    all_problems = [chr(ord('A') + i) for i in range(13)]

    # 定义CSV的列顺序
    base_columns = ["排名", "真实姓名", "牛客名", "AC数(总)", "罚时(ms)", "学校", "参赛状态"]
    problem_columns = [f"题{p}" for p in all_problems]
    stat_columns = ["AC数(详)", "WA数(详)", "未提交数"]

    all_columns = base_columns + problem_columns + stat_columns

    # 用于存储所有比赛的数据
    all_data = []

    for cid in contest_ids:
        print(f"\n{'='*60}")
        print(f"正在处理比赛 {cid}...")
        print(f"{'='*60}")

        rank_data, basic = fetch_rank(cid)

        print(f"比赛 {cid}: rankCount={basic.get('rankCount')}  返回={len(rank_data)}")

        # 构建包含题目状态的DataFrame
        df = build_school_df(rank_data, cid)

        # 添加比赛标题
        all_data.append([f"比赛 {cid}（本校成绩单）"])
        all_data.append([f"题目范围: {' '.join(all_problems)}"])
        all_data.append([])

        # 分离已参赛和未参赛的选手
        participated_mask = df['参赛状态'] == '已参赛' if '参赛状态' in df.columns else pd.Series([False]*len(df))
        participated_df = df[participated_mask].copy()
        not_participated_df = df[~participated_mask].copy() if '参赛状态' in df.columns else pd.DataFrame()

        # 为已参赛选手添加缺失的列
        for col in all_columns:
            if col not in participated_df.columns:
                participated_df[col] = ""

        # 已参赛选手排序：按 AC 降序，罚时升序
        if not participated_df.empty:
            participated_df = participated_df.sort_values(
                by=["AC数(总)", "罚时(ms)"],
                ascending=[False, True]
            )

        # 未参赛选手只保留基本信息列
        not_participated_cols = ["真实姓名", "牛客名"]

        # 添加已参赛选手
        all_data.append(["【已参赛选手】"])
        if not participated_df.empty:
            # 添加表头
            all_data.append(all_columns)
            # 添加数据行
            for row in participated_df[all_columns].itertuples(index=False):
                # 将空字符串替换为空白显示，并将整数列转换为整数格式
                row_list = []
                for cell in row:
                    if isinstance(cell, float) and cell.is_integer():
                        row_list.append(int(cell))  # 将浮点数转换为整数
                    elif cell == '':
                        row_list.append('')
                    else:
                        row_list.append(cell)
                all_data.append(row_list)
        else:
            all_data.append(["暂无本校选手参赛"])

        all_data.append([])

        # 添加未参赛选手
        all_data.append(["【未参赛选手】"])
        if not not_participated_df.empty:
            # 未参赛选手只显示姓名和牛客名，没有题目列
            all_data.append(not_participated_cols)
            for row in not_participated_df[not_participated_cols].itertuples(index=False):
                all_data.append(list(row))
        else:
            all_data.append(["所有选手均已参赛"])

        # 分隔空行
        all_data.append([])
        all_data.append([])
        all_data.append([])

    # 导出CSV前，确保所有数值为整数
    final_rows = []
    for row in all_data:
        new_row = []
        for cell in row:
            if isinstance(cell, float) and cell.is_integer():
                new_row.append(int(cell))
            else:
                new_row.append(cell)
        final_rows.append(new_row)

    # 导出CSV
    out_df = pd.DataFrame(final_rows)
    filename = "牛客寒假训练营.csv"
    out_df.to_csv(filename, index=False, header=False, encoding="utf-8-sig")
    print(f"\n✅ 已导出：{filename}")


def get_single_user_status():
    """单独获取某个用户的提交状态（用于测试）"""
    contest_id = "120566"
    username = "南与江"

    print(f"\n🔍 单独获取用户 [{username}] 在比赛 [{contest_id}] 中的提交状态...")

    api_data = fetch_user_submissions(contest_id, username)

    if api_data:
        result = get_problem_status(api_data)

        if result and result[0]:
            status_dict, status_list = result

            # 打印汇总
            problems = list(status_dict.keys())

            print("\n" + "="*80)
            print(f"📊 用户 [{username}] 的题目提交状态")
            print("="*80)

            # 打印题号行
            print("题号:", end=" ")
            for p in problems:
                print(f"  {p}  ", end="")
            print()

            # 打印分隔线
            print("     " + "-" * (len(problems) * 4))

            # 打印状态行
            print("状态:", end=" ")
            for s in status_list:
                if s == 'AC':
                    print(f" \033[92m{s}\033[0m ", end="")  # 绿色
                elif s == 'WA':
                    print(f" \033[91m{s}\033[0m ", end="")  # 红色
                else:
                    print(f"    ", end="")  # 未提交显示空白
            print()

            print("="*80)
            ac_count = status_list.count('AC')
            wa_count = status_list.count('WA')
            unsubmitted_count = status_list.count('')
            print(f"📈 统计: AC: {ac_count}  |  WA: {wa_count}  |  未提交: {unsubmitted_count}")
            print("="*80)

            # 保存为CSV
            filename = save_user_status_to_csv(status_dict, username, contest_id)
            print(f"✅ 已保存到: {filename}")


if __name__ == "__main__":
    # 主功能：获取所有比赛的成绩单
    solve()

    # 可选：单独测试某个用户的提交状态
    # get_single_user_status()


正在处理比赛 120561...
比赛 120561: rankCount=6822  返回=6822

📋 本题比赛题目范围: A B C D E F G H I J K L M
  正在获取 南与江 的提交状态...
  正在获取 lihanyu 的提交状态...
  正在获取 Patience019 的提交状态...
  正在获取 大鱼儿半生不熟 的提交状态...
  正在获取 caimingzheng 的提交状态...
  正在获取 真想一下子睡过去 的提交状态...
  正在获取 幼儿园校霸的校霸 的提交状态...
  正在获取 HHHzzzyyy 的提交状态...
  正在获取 Makima667 的提交状态...
  正在获取 芭六 的提交状态...
  正在获取 ekko___ 的提交状态...
  正在获取 坠姆哈尼 的提交状态...
  正在获取 万月838326 的提交状态...
  正在获取 2421211576 的提交状态...
  正在获取 yubeibei 的提交状态...
  正在获取 keshao 的提交状态...
  正在获取 kyhnb66 的提交状态...
  正在获取 谬儒 的提交状态...
  正在获取 ccs821631 的提交状态...
  正在获取 alayaf 的提交状态...
  正在获取 枣虾鱼 的提交状态...
  正在获取 轩屁屁的核桃 的提交状态...
  正在获取 GoldenRipple 的提交状态...
  正在获取 柯大人 的提交状态...
  正在获取 水悲 的提交状态...
  正在获取 牛客891390731号 的提交状态...
  正在获取 牛客390356941号 的提交状态...

正在处理比赛 120562...
比赛 120562: rankCount=6356  返回=6356

📋 本题比赛题目范围: A B C D E F G H I J K L M
  正在获取 南与江 的提交状态...
  正在获取 lihanyu 的提交状态...
  正在获取 芭六 的提交状态...
  正在获取 Patience019 的提交状态...
  正在获取 大鱼儿半生不熟 的提交状态...
  正在获取 caimingzheng 的提交状态...
  正在获取 幼儿园校霸的校霸 的提交状